In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/DS")

for item in base.iterdir():
    print(repr(item.name))

'NF_ToN_IoT_v3.'


In [ ]:
import pandas as pd
from pathlib import Path

matches = list(
    Path("/content/drive/MyDrive/DS").rglob("NF-ToN-IoT-v3.csv")
)

if len(matches) == 1:
    dataset_path = matches[0]
    print("File:", dataset_path)

    preview = pd.read_csv(dataset_path, nrows=5)
    display(preview)
else:
    print("Expected one matching file. Found:", len(matches))
    for path in matches:
        print(path)

File: /content/drive/MyDrive/DS/NF_ToN_IoT_v3./NF-ToN-IoT-v3.csv


,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1556028590618,1556028590618,192.168.1.32,41315,192.168.1.169,10010,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
1,1556028590618,1556028590621,192.168.1.32,41315,192.168.1.194,49400,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
2,1556028590618,1556028590621,192.168.1.32,41315,192.168.1.193,8008,6,161,44,1,...,0,0,0,0,0,0,0,0,1,scanning
3,1556028590624,1556028590628,192.168.1.32,41315,192.168.1.194,85,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
4,1556028590624,1556028590624,192.168.1.32,41317,192.168.1.180,1056,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign


In [ ]:
preview.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 55 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   FLOW_START_MILLISECONDS      5 non-null      int64 
 1   FLOW_END_MILLISECONDS        5 non-null      int64 
 2   IPV4_SRC_ADDR                5 non-null      object
 3   L4_SRC_PORT                  5 non-null      int64 
 4   IPV4_DST_ADDR                5 non-null      object
 5   L4_DST_PORT                  5 non-null      int64 
 6   PROTOCOL                     5 non-null      int64 
 7   L7_PROTO                     5 non-null      int64 
 8   IN_BYTES                     5 non-null      int64 
 9   IN_PKTS                      5 non-null      int64 
 10  OUT_BYTES                    5 non-null      int64 
 11  OUT_PKTS                     5 non-null      int64 
 12  TCP_FLAGS                    5 non-null      int64 
 13  CLIENT_TCP_FLAGS             5 non-null

In [ ]:
total_rows = 0
attack_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    dataset_path,
    usecols=["Attack"],
    dtype={"Attack": "string"},
    chunksize=100_000
):
    total_rows += len(chunk)

    counts = chunk["Attack"].value_counts(dropna=False)
    attack_counts = attack_counts.add(counts, fill_value=0)

attack_counts = attack_counts.astype("int64").sort_values(ascending=False)

print("Total rows:", f"{total_rows:,}")
print("Total columns:", len(preview.columns))

distribution = attack_counts.rename_axis("Attack").to_frame("Count")
distribution["Percentage"] = (
    distribution["Count"] / total_rows * 100
).round(3)

display(distribution)

Total rows: 27,520,260
Total columns: 55


,Count,Percentage
Attack,,
Benign,16792214,61.018
ddos,4141256,15.048
xss,2834435,10.299
password,1594777,5.795
scanning,1358977,4.938
injection,381777,1.387
dos,203456,0.739
Backdoor,203384,0.739
mitm,6013,0.022


In [ ]:
import csv
from collections import Counter

# Read only the original header
with open(dataset_path, encoding="utf-8-sig", newline="") as file:
    headers = next(csv.reader(file))

spaces = [repr(name) for name in headers if name != name.strip()]
duplicates = [
    name for name, count in Counter(headers).items() if count > 1
]

print("Number of columns:", len(headers))
print("Headers with extra spaces:", spaces)
print("Repeated column names:", duplicates)

Number of columns: 55
Headers with extra spaces: []
Repeated column names: []


no issue


In [ ]:
missing_counts = pd.Series(0, index=headers, dtype="int64")
rows_checked = 0

for chunk in pd.read_csv(
    dataset_path,
    dtype="string",
    chunksize=100_000
):
    missing_counts += chunk.isna().sum()
    rows_checked += len(chunk)

    if rows_checked % 5_000_000 == 0:
        print(f"Checked {rows_checked:,} rows", flush=True)

missing_summary = pd.DataFrame({
    "Missing count": missing_counts,
    "Missing percentage": missing_counts / rows_checked * 100
})

print(f"Finished: {rows_checked:,} rows checked")
display(missing_summary.sort_values("Missing count", ascending=False))

Checked 5,000,000 rows
Checked 10,000,000 rows
Checked 15,000,000 rows
Checked 20,000,000 rows
Checked 25,000,000 rows
Finished: 27,520,260 rows checked


,Missing count,Missing percentage
FLOW_START_MILLISECONDS,0,0.0
FLOW_END_MILLISECONDS,0,0.0
IPV4_SRC_ADDR,0,0.0
L4_SRC_PORT,0,0.0
IPV4_DST_ADDR,0,0.0
L4_DST_PORT,0,0.0
PROTOCOL,0,0.0
L7_PROTO,0,0.0
IN_BYTES,0,0.0
IN_PKTS,0,0.0


In [ ]:
import numpy as np

text_columns = {"IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Attack"}
numeric_columns = [c for c in headers if c not in text_columns]

audit = {
    c: {
        "Count": 0, "Mean": 0.0, "M2": 0.0,
        "Min": np.inf, "Max": -np.inf,
        "Non-numeric": 0, "Infinite": 0, "Negative": 0,
        "Example rows": []
    }
    for c in numeric_columns
}

rows_scanned = 0

for chunk in pd.read_csv(
    dataset_path, chunksize=100_000, low_memory=False
):
    for column in numeric_columns:
        original = chunk[column]

        # Temporary numeric interpretation for inspection only
        values = pd.to_numeric(original, errors="coerce").astype("float64")

        non_numeric = original.notna() & values.isna()
        infinite = np.isinf(values)
        negative = values.notna() & ~infinite & (values < 0)

        result = audit[column]
        result["Non-numeric"] += int(non_numeric.sum())
        result["Infinite"] += int(infinite.sum())
        result["Negative"] += int(negative.sum())

        suspicious = non_numeric | infinite | negative
        remaining = 3 - len(result["Example rows"])
        result["Example rows"].extend(
            (chunk.index[suspicious][:remaining] + 1).tolist()
        )

        # Statistics use finite numeric values only
        valid = values[values.notna() & ~infinite]
        n = len(valid)

        if n:
            previous_n = result["Count"]
            combined_n = previous_n + n
            chunk_mean = valid.mean()
            difference = chunk_mean - result["Mean"]

            result["M2"] += (
                ((valid - chunk_mean) ** 2).sum()
                + difference**2 * previous_n * n / combined_n
            )
            result["Mean"] += difference * n / combined_n
            result["Count"] = combined_n
            result["Min"] = min(result["Min"], valid.min())
            result["Max"] = max(result["Max"], valid.max())

    rows_scanned += len(chunk)

    if rows_scanned % 1_000_000 == 0:
        print(f"Checked {rows_scanned:,} rows", flush=True)

audit_summary = pd.DataFrame(audit).T
audit_summary["Std"] = [
    np.sqrt(r["M2"] / (r["Count"] - 1))
    if r["Count"] > 1 else np.nan
    for r in audit.values()
]

print(f"Finished: {rows_scanned:,} rows checked")

issue_columns = ["Non-numeric", "Infinite", "Negative"]
flagged = audit_summary[issue_columns].sum(axis=1) > 0

display(audit_summary.loc[
    flagged, issue_columns + ["Example rows"]
])

Checked 1,000,000 rows
Checked 2,000,000 rows
Checked 3,000,000 rows
Checked 4,000,000 rows
Checked 5,000,000 rows
Checked 6,000,000 rows
Checked 7,000,000 rows
Checked 8,000,000 rows
Checked 9,000,000 rows
Checked 10,000,000 rows
Checked 11,000,000 rows
Checked 12,000,000 rows
Checked 13,000,000 rows
Checked 14,000,000 rows
Checked 15,000,000 rows
Checked 16,000,000 rows
Checked 17,000,000 rows
Checked 18,000,000 rows
Checked 19,000,000 rows
Checked 20,000,000 rows
Checked 21,000,000 rows
Checked 22,000,000 rows
Checked 23,000,000 rows
Checked 24,000,000 rows
Checked 25,000,000 rows
Checked 26,000,000 rows
Checked 27,000,000 rows
Finished: 27,520,260 rows checked


,Non-numeric,Infinite,Negative,Example rows


In [ ]:
features = [
    "IN_BYTES",
    "OUT_BYTES",
    "IN_PKTS",
    "OUT_PKTS",
    "FLOW_DURATION_MILLISECONDS"
]

display(audit_summary.loc[
    features, ["Min", "Max", "Mean", "Std"]
])

,Min,Max,Mean,Std
IN_BYTES,4.0,79846944.0,454.481807,38862.084632
OUT_BYTES,0.0,70221306.0,372.271106,27623.223859
IN_PKTS,1.0,1220131.0,4.049334,624.165065
OUT_PKTS,0.0,1350048.0,1.909364,376.664103
FLOW_DURATION_MILLISECONDS,1.0,120999.0,161.929829,1707.517645


In [ ]:
constant_mask = (
    (audit_summary["Count"] == rows_scanned)
    & (audit_summary["Min"] == audit_summary["Max"])
)

display(audit_summary.loc[
    constant_mask, ["Min", "Max"]
])

,Min,Max


No numerical column had the same value in every row, so none is flagged for removal as a constant feature.

In [ ]:
columns_to_check = [
    "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS",
    "L4_SRC_PORT", "L4_DST_PORT",
    "SHORTEST_FLOW_PKT", "LONGEST_FLOW_PKT",
    "Label", "Attack"
]

check_counts = {}
check_examples = {}
checked = 0

for chunk in pd.read_csv(
    dataset_path,
    usecols=columns_to_check,
    chunksize=100_000
):
    checks = {
        "End before start":
            chunk["FLOW_END_MILLISECONDS"]
            < chunk["FLOW_START_MILLISECONDS"],

        "Source port outside 0–65535":
            ~chunk["L4_SRC_PORT"].between(0, 65535),

        "Destination port outside 0–65535":
            ~chunk["L4_DST_PORT"].between(0, 65535),

        "Smallest packet larger than largest":
            chunk["SHORTEST_FLOW_PKT"]
            > chunk["LONGEST_FLOW_PKT"],

        "Label outside 0 or 1":
            ~chunk["Label"].isin([0, 1]),

        "Label and Attack disagree":
            chunk["Label"].isin([0, 1]) & (
                (chunk["Label"] == 0)
                != (chunk["Attack"] == "Benign")
            )
    }

    for name, mask in checks.items():
        check_counts[name] = check_counts.get(name, 0) + int(mask.sum())
        examples = check_examples.setdefault(name, [])
        examples.extend(
            (chunk.index[mask][:3 - len(examples)] + 1).tolist()
        )

    checked += len(chunk)

    if checked % 5_000_000 == 0:
        print(f"Checked {checked:,} rows", flush=True)

consistency_summary = pd.DataFrame({
    "Count": check_counts,
    "Example rows": check_examples
})
consistency_summary["Percentage"] = (
    consistency_summary["Count"] / checked * 100
)

print(f"Finished: {checked:,} rows checked")
display(consistency_summary)

Checked 5,000,000 rows
Checked 10,000,000 rows
Checked 15,000,000 rows
Checked 20,000,000 rows
Checked 25,000,000 rows
Finished: 27,520,260 rows checked


,Count,Example rows,Percentage
End before start,0,[],0.0
Source port outside 0–65535,0,[],0.0
Destination port outside 0–65535,0,[],0.0
Smallest packet larger than largest,0,[],0.0
Label outside 0 or 1,0,[],0.0
Label and Attack disagree,0,[],0.0


In [ ]:
%pip -q install duckdb

import duckdb

with duckdb.connect() as con:
    con.execute("SET memory_limit = '2GB'")
    con.execute("SET threads = 2")
    con.execute("SET temp_directory = '/content/duplicate_check_tmp'")
    con.execute("SET preserve_insertion_order = false")

    print("Checking the whole file for duplicates...", flush=True)

    total, extra_copies = con.execute("""
        SELECT
            SUM(copies),
            COALESCE(SUM(copies - 1), 0)
        FROM (
            SELECT *, COUNT(*) AS copies
            FROM read_csv(?, header = true, all_varchar = true)
            GROUP BY ALL
        )
    """, [str(dataset_path)]).fetchone()

total = int(total)
extra_copies = int(extra_copies)

print(f"Rows checked: {total:,}")
print(f"Extra duplicate copies: {extra_copies:,}")
print(f"Duplicate percentage: {extra_copies / total * 100:.4f}%")

Checking the whole file for duplicates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows checked: 27,520,260
Extra duplicate copies: 1,816,137
Duplicate percentage: 6.5993%


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

output_folder = Path("/content/drive/MyDrive/DS/audit_results")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / f"{Path(dataset_path).stem}_duplicates.csv"

# Quote file paths and column names for SQL
def sql_text(value):
    return "'" + str(value).replace("'", "''") + "'"

def sql_column(name):
    return '"' + name.replace('"', '""') + '"'

columns = ", ".join(sql_column(c) for c in headers)

matching = " AND ".join(
    f"n.{sql_column(c)} IS NOT DISTINCT FROM d.{sql_column(c)}"
    for c in headers
)

source = (
    f"read_csv({sql_text(dataset_path)}, "
    "header=true, all_varchar=true, parallel=false)"
)

with duckdb.connect() as con:
    con.execute("SET memory_limit = '2GB'")
    con.execute("SET threads = 1")
    con.execute("SET preserve_insertion_order = true")
    con.execute("SET temp_directory = '/content/duplicate_details_tmp'")

    print("Finding records that repeat...", flush=True)

    con.execute(f"""
        CREATE TEMP TABLE repeated_records AS
        SELECT {columns}, COUNT(*) AS occurrence_count
        FROM {source}
        GROUP BY ALL
        HAVING COUNT(*) > 1
    """)

    print("Finding their row numbers and saving the results...", flush=True)

    con.execute(f"""
        COPY (
            WITH numbered AS (
                SELECT ROW_NUMBER() OVER () AS data_row, *
                FROM {source}
            )
            SELECT n.data_row, d.occurrence_count, n.* EXCLUDE (data_row)
            FROM numbered AS n
            JOIN repeated_records AS d ON {matching}
        )
        TO {sql_text(output_file)}
        (FORMAT CSV, HEADER true)
    """)

print("Saved:", output_file)

# Show only a preview; the saved file contains all matching records
display(pd.read_csv(output_file, nrows=10))

Finding records that repeat...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Finding their row numbers and saving the results...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: /content/drive/MyDrive/DS/audit_results/NF-ToN-IoT-v3_duplicates.csv


,data_row,occurrence_count,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,9979433,2,1556129989364,1556129989364,192.168.1.190,53,172.17.0.3,59819,17,5,...,0,0,0,0,0,0,0,0,0,Benign
1,9979435,2,1556129989364,1556129989364,192.168.1.190,53,172.17.0.3,59819,17,5,...,0,0,0,0,0,0,0,0,0,Benign
2,9990123,2,1556131021955,1556131021955,192.168.1.190,53,172.17.0.3,37123,17,5,...,0,0,0,0,0,0,0,0,0,Benign
3,9990124,2,1556131021955,1556131021955,192.168.1.190,53,172.17.0.3,37123,17,5,...,0,0,0,0,0,0,0,0,0,Benign
4,25165916,2,1556434429730,1556434429768,192.168.1.180,123,203.122.222.157,123,17,9,...,0,0,0,0,0,0,0,0,0,Benign
5,25165917,2,1556434429730,1556434429768,192.168.1.180,123,203.122.222.157,123,17,9,...,0,0,0,0,0,0,0,0,0,Benign
6,25165918,2,1556434429730,1556434429796,192.168.1.180,123,27.124.125.251,123,17,9,...,0,0,0,0,0,0,0,0,0,Benign
7,25165919,2,1556434429730,1556434429754,192.168.1.180,123,116.66.161.7,123,17,9,...,0,0,0,0,0,0,0,0,0,Benign
8,25165920,2,1556434429730,1556434429796,192.168.1.180,123,27.124.125.251,123,17,9,...,0,0,0,0,0,0,0,0,0,Benign
9,25165921,2,1556434429730,1556434429754,192.168.1.180,123,116.66.161.7,123,17,9,...,0,0,0,0,0,0,0,0,0,Benign


In [ ]:
grouped_file = output_file.with_name(
    output_file.stem + "_grouped.csv"
)

with duckdb.connect() as con:
    con.execute("SET memory_limit = '2GB'")
    con.execute("SET threads = 1")
    con.execute("SET temp_directory = '/content/duplicate_groups_tmp'")

    con.execute(f"""
        COPY (
            SELECT
                STRING_AGG(
                    data_row, ', '
                    ORDER BY CAST(data_row AS BIGINT)
                ) AS matching_row_numbers,
                occurrence_count,
                {columns}
            FROM read_csv(
                {sql_text(output_file)},
                header=true,
                all_varchar=true
            )
            GROUP BY ALL
        )
        TO {sql_text(grouped_file)}
        (FORMAT CSV, HEADER true)
    """)

display(pd.read_csv(
    grouped_file,
    usecols=["matching_row_numbers", "occurrence_count"],
    nrows=10
))

print("All matching groups saved to:", grouped_file)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,matching_row_numbers,occurrence_count
0,"9979433, 9979435",2
1,"9990123, 9990124",2
2,"25165916, 25165917",2
3,"25165918, 25165920",2
4,"25165919, 25165921",2
5,"25165922, 25165924",2
6,"25165923, 25165925",2
7,"25165926, 25165931",2
8,"25165927, 25165932",2
9,"25165928, 25165933",2


All matching groups saved to: /content/drive/MyDrive/DS/audit_results/NF-ToN-IoT-v3_duplicates_grouped.csv


In [ ]:
features = [
    "IN_BYTES",
    "OUT_BYTES",
    "IN_PKTS",
    "OUT_PKTS",
    "FLOW_DURATION_MILLISECONDS"
]

expressions = ", ".join(
    f"quantile_cont(CAST({sql_column(c)} AS DOUBLE), "
    f"[0.25, 0.50, 0.75])"
    for c in features
)

print("Calculating quartiles for the whole dataset...", flush=True)

with duckdb.connect() as con:
    con.execute("SET memory_limit = '2GB'")
    con.execute("SET threads = 1")

    quartiles = con.execute(f"""
        SELECT {expressions}
        FROM read_csv(?, header=true, all_varchar=true)
    """, [str(dataset_path)]).fetchone()

iqr_summary = pd.DataFrame(
    list(quartiles),
    index=features,
    columns=["Q1", "Median", "Q3"]
)

iqr_summary["IQR"] = iqr_summary["Q3"] - iqr_summary["Q1"]

iqr_summary["Lower limit"] = (
    iqr_summary["Q1"] - 1.5 * iqr_summary["IQR"]
)

iqr_summary["Upper limit"] = (
    iqr_summary["Q3"] + 1.5 * iqr_summary["IQR"]
)

display(iqr_summary)

Calculating quartiles for the whole dataset...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Q1,Median,Q3,IQR,Lower limit,Upper limit
IN_BYTES,48.0,67.0,120.0,72.0,-60.0,228.0
OUT_BYTES,0.0,40.0,65.0,65.0,-97.5,162.5
IN_PKTS,1.0,1.0,2.0,1.0,-0.5,3.5
OUT_PKTS,0.0,1.0,1.0,1.0,-1.5,2.5
FLOW_DURATION_MILLISECONDS,1.0,1.0,4.0,3.0,-3.5,8.5


In [ ]:
outlier_file = output_folder / f"{Path(dataset_path).stem}_iqr_rows.csv"

outlier_counts = pd.Series(0, index=features, dtype="int64")
example_rows = {c: [] for c in features}

outlier_rows_checked = 0
flagged_rows = 0
write_header = True

with outlier_file.open("w", newline="") as results:
    for chunk in pd.read_csv(
        dataset_path,
        usecols=features + ["Label", "Attack"],
        chunksize=100_000
    ):
        # Check each value against its feature's limits
        flags = (
            chunk[features].lt(
                iqr_summary["Lower limit"], axis="columns"
            )
            |
            chunk[features].gt(
                iqr_summary["Upper limit"], axis="columns"
            )
        )

        outlier_counts += flags.sum()

        # Keep a few row numbers for the summary
        for c in features:
            remaining = 5 - len(example_rows[c])
            example_rows[c].extend(
                (chunk.index[flags[c]][:remaining] + 1).tolist()
            )

        # Save every row flagged in at least one feature
        flagged = chunk.loc[flags.any(axis=1)].copy()
        flagged.insert(0, "data_row", flagged.index + 1)

        for c in features:
            flagged[f"{c}_IQR_flag"] = flags.loc[flagged.index, c]

        flagged.to_csv(results, index=False, header=write_header)
        write_header = False

        flagged_rows += len(flagged)
        outlier_rows_checked += len(chunk)

        if outlier_rows_checked % 5_000_000 == 0:
            print(f"Checked {outlier_rows_checked:,} rows", flush=True)

outlier_summary = iqr_summary.copy()
outlier_summary["Flagged count"] = outlier_counts
outlier_summary["Flagged percentage"] = (
    outlier_counts / outlier_rows_checked * 100
)
outlier_summary["Example row numbers"] = pd.Series(example_rows)

outlier_summary.to_csv(
    output_folder / f"{Path(dataset_path).stem}_iqr_summary.csv",
    index_label="Feature"
)

display(outlier_summary)
print(f"Rows flagged in at least one feature: {flagged_rows:,}")
print("All flagged rows saved to:", outlier_file)

Checked 5,000,000 rows
Checked 10,000,000 rows
Checked 15,000,000 rows
Checked 20,000,000 rows
Checked 25,000,000 rows


,Q1,Median,Q3,IQR,Lower limit,Upper limit,Flagged count,Flagged percentage,Example row numbers
IN_BYTES,48.0,67.0,120.0,72.0,-60.0,228.0,6088280,22.122901,"[1074, 1383, 1384, 1547, 1573]"
OUT_BYTES,0.0,40.0,65.0,65.0,-97.5,162.5,4611672,16.757371,"[1383, 1547, 1573, 1600, 1619]"
IN_PKTS,1.0,1.0,2.0,1.0,-0.5,3.5,4952626,17.996291,"[1383, 1384, 1547, 1573, 1600]"
OUT_PKTS,0.0,1.0,1.0,1.0,-1.5,2.5,4659346,16.930603,"[1383, 1384, 1547, 1573, 1600]"
FLOW_DURATION_MILLISECONDS,1.0,1.0,4.0,3.0,-3.5,8.5,5872001,21.337011,"[661, 724, 735, 736, 737]"


Rows flagged in at least one feature: 7,992,507
All flagged rows saved to: /content/drive/MyDrive/DS/audit_results/NF-ToN-IoT-v3_iqr_rows.csv


In [ ]:
measurement_features = features + [
    "DURATION_IN", "DURATION_OUT",
    "LONGEST_FLOW_PKT", "SHORTEST_FLOW_PKT",
    "MIN_IP_PKT_LEN", "MAX_IP_PKT_LEN",
    "SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES",
    "RETRANSMITTED_IN_BYTES", "RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_BYTES", "RETRANSMITTED_OUT_PKTS",
    "SRC_TO_DST_AVG_THROUGHPUT", "DST_TO_SRC_AVG_THROUGHPUT",
    "NUM_PKTS_UP_TO_128_BYTES",
    "NUM_PKTS_128_TO_256_BYTES",
    "NUM_PKTS_256_TO_512_BYTES",
    "NUM_PKTS_512_TO_1024_BYTES",
    "NUM_PKTS_1024_TO_1514_BYTES",
    "TCP_WIN_MAX_IN", "TCP_WIN_MAX_OUT",
    "DNS_TTL_ANSWER",
    "SRC_TO_DST_IAT_MIN", "SRC_TO_DST_IAT_MAX",
    "SRC_TO_DST_IAT_AVG", "SRC_TO_DST_IAT_STDDEV",
    "DST_TO_SRC_IAT_MIN", "DST_TO_SRC_IAT_MAX",
    "DST_TO_SRC_IAT_AVG", "DST_TO_SRC_IAT_STDDEV"
]

cache_path = Path("/content") / (
    Path(dataset_path).stem + "_measurements.parquet"
)

all_iqr_summary = iqr_summary[["Q1", "Median", "Q3"]].copy()

with duckdb.connect() as con:
    con.execute("SET memory_limit = '2GB'")
    con.execute("SET threads = 1")
    con.execute("SET preserve_insertion_order = true")

    if not cache_path.exists():
        print("Preparing the temporary copy...", flush=True)

        selected_columns = ", ".join(
            sql_column(c)
            for c in measurement_features + ["Label", "Attack"]
        )

        building_path = cache_path.with_suffix(".building.parquet")

        con.execute(f"""
            COPY (
                SELECT
                    ROW_NUMBER() OVER () AS data_row,
                    {selected_columns}
                FROM read_csv(
                    {sql_text(dataset_path)},
                    header=true,
                    all_varchar=true,
                    parallel=false
                )
            )
            TO {sql_text(building_path)}
            (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 1)
        """)

        building_path.replace(cache_path)

    remaining = [
        c for c in measurement_features
        if c not in all_iqr_summary.index
    ]

    for start in range(0, len(remaining), 5):
        batch = remaining[start:start + 5]
        print(f"Calculating batch {start // 5 + 1}/6...", flush=True)

        expressions = ", ".join(
            f"quantile_cont(CAST({sql_column(c)} AS DOUBLE), "
            "[0.25, 0.50, 0.75])"
            for c in batch
        )

        values = con.execute(f"""
            SELECT {expressions}
            FROM read_parquet(?)
        """, [str(cache_path)]).fetchone()

        for c, quartiles in zip(batch, values):
            all_iqr_summary.loc[c, ["Q1", "Median", "Q3"]] = quartiles

all_iqr_summary["IQR"] = (
    all_iqr_summary["Q3"] - all_iqr_summary["Q1"]
)
all_iqr_summary["Lower limit"] = (
    all_iqr_summary["Q1"] - 1.5 * all_iqr_summary["IQR"]
)
all_iqr_summary["Upper limit"] = (
    all_iqr_summary["Q3"] + 1.5 * all_iqr_summary["IQR"]
)
all_iqr_summary["Needs separate review"] = (
    all_iqr_summary["IQR"] == 0
)

all_iqr_summary.to_csv(
    output_folder / f"{Path(dataset_path).stem}_all_iqr_limits.csv",
    index_label="Feature"
)

display(all_iqr_summary)

Preparing the temporary copy...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Calculating batch 1/6...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Calculating batch 2/6...


...

Calculating batch 5/6...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Calculating batch 6/6...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Q1,Median,Q3,IQR,Lower limit,Upper limit,Needs separate review
IN_BYTES,48.0,67.0,120.0,72.0,-60.0,228.0,False
OUT_BYTES,0.0,40.0,65.0,65.0,-97.5,162.5,False
IN_PKTS,1.0,1.0,2.0,1.0,-0.5,3.5,False
OUT_PKTS,0.0,1.0,1.0,1.0,-1.5,2.5,False
FLOW_DURATION_MILLISECONDS,1.0,1.0,4.0,3.0,-3.5,8.5,False
DURATION_IN,0.0,0.0,0.0,0.0,0.0,0.0,True
DURATION_OUT,0.0,0.0,0.0,0.0,0.0,0.0,True
LONGEST_FLOW_PKT,44.0,60.0,89.0,45.0,-23.5,156.5,False
SHORTEST_FLOW_PKT,44.0,52.0,52.0,8.0,32.0,64.0,False
MIN_IP_PKT_LEN,0.0,40.0,52.0,52.0,-78.0,130.0,False


In [ ]:
import pickle

names = [
    "dataset_path", "headers", "attack_counts", "total_rows",
    "missing_summary", "audit_summary", "consistency_summary",
    "iqr_summary", "all_iqr_summary", "outlier_summary",
    "features", "measurement_features"
]

checkpoint = {name: globals()[name] for name in names if name in globals()}

with open("/content/drive/MyDrive/DS/audit_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

print("Progress saved.")

Progress saved.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pickle
import pandas as pd
import numpy as np
from pathlib import Path

with open("/content/drive/MyDrive/DS/audit_checkpoint.pkl", "rb") as f:
    checkpoint = pickle.load(f)

globals().update(checkpoint)

print("Progress restored.")
print("Dataset:", Path(dataset_path).name)
print("Features with IQR limits:",
      len(checkpoint.get("all_iqr_summary", [])))

Mounted at /content/drive
Progress restored.
Dataset: NF-ToN-IoT-v3.csv
Features with IQR limits: 35


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
from collections import Counter

output_folder = Path("/content/drive/MyDrive/DS/audit_results")
output_folder.mkdir(parents=True, exist_ok=True)
stem = Path(dataset_path).stem

limits = all_iqr_summary.loc[measurement_features].copy()
regular = limits.index[limits["IQR"] > 0].tolist()
zero_iqr = limits.index[limits["IQR"] == 0].tolist()

counts = pd.Series(0, index=measurement_features, dtype="int64")
zeros = counts.copy()
examples = {c: [] for c in measurement_features}
class_totals, class_flags, class_variations = Counter(), Counter(), Counter()
checked = 0

details_path = output_folder / f"{stem}_measurement_flags.parquet"
partial_path = details_path.with_suffix(".partial.parquet")
cache_path = Path("/content") / f"{stem}_measurements.parquet"

# Reuse yesterday's temporary copy if it still exists
if cache_path.exists():
    batches = (
        batch.to_pandas()
        for batch in pq.ParquetFile(cache_path).iter_batches(
            batch_size=100_000,
            columns=measurement_features + ["Label", "Attack"]
        )
    )
else:
    batches = pd.read_csv(
        dataset_path,
        usecols=measurement_features + ["Label", "Attack"],
        chunksize=100_000
    )

writer = None
try:
    for chunk in batches:
        chunk[measurement_features] = chunk[measurement_features].astype("float64")
        chunk["Attack"] = chunk["Attack"].astype("string")
        chunk["Label"] = chunk["Label"].astype("int8")
        chunk.insert(0, "data_row",
                     np.arange(checked + 1, checked + len(chunk) + 1))

        flags = (
            chunk[measurement_features].lt(limits["Lower limit"], axis="columns")
            |
            chunk[measurement_features].gt(limits["Upper limit"], axis="columns")
        )

        counts += flags.sum()
        zeros += chunk[measurement_features].eq(0).sum()

        for c in measurement_features:
            remaining = 3 - len(examples[c])
            examples[c].extend(
                chunk.loc[flags[c], "data_row"].head(remaining).tolist()
            )

        regular_mask = flags[regular].any(axis=1)
        variation_mask = flags[zero_iqr].any(axis=1)

        class_totals.update(chunk["Attack"])
        class_flags.update(chunk.loc[regular_mask, "Attack"])
        class_variations.update(chunk.loc[variation_mask, "Attack"])

        # Save all identified records with values and feature flags
        selected = flags.any(axis=1)
        if selected.any():
            records = chunk.loc[selected].copy()
            records = pd.concat([
                records,
                flags.loc[selected].add_suffix("_screen_flag")
            ], axis=1)

            table = pa.Table.from_pandas(records, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(
                    str(partial_path), table.schema, compression="zstd"
                )
            writer.write_table(table)

        checked += len(chunk)
        if checked % 5_000_000 == 0:
            print(f"Checked {checked:,} rows", flush=True)
finally:
    if writer is not None:
        writer.close()

if writer is not None:
    partial_path.replace(details_path)

all_screen_summary = limits.copy()
all_screen_summary["Interpretation"] = np.where(
    limits["IQR"] > 0,
    "Potential IQR outlier",
    "Different from common value; IQR=0"
)
all_screen_summary["Flagged count"] = counts
all_screen_summary["Flagged %"] = counts / checked * 100
all_screen_summary["Zero %"] = zeros / checked * 100
all_screen_summary["Example rows"] = pd.Series(examples)

class_screen_summary = pd.DataFrame({
    "Total flows": pd.Series(class_totals),
    "IQR-flagged flows": pd.Series(class_flags),
    "IQR=0 variation flows": pd.Series(class_variations)
}).fillna(0).astype("int64")

class_screen_summary["IQR-flagged % within class"] = (
    class_screen_summary["IQR-flagged flows"]
    / class_screen_summary["Total flows"] * 100
)

all_screen_summary.to_csv(output_folder / f"{stem}_measurement_summary.csv")
class_screen_summary.to_csv(output_folder / f"{stem}_flags_by_class.csv")

# Save progress automatically
checkpoint.update({
    "all_screen_summary": all_screen_summary,
    "class_screen_summary": class_screen_summary,
    "details_path": details_path
})
with open("/content/drive/MyDrive/DS/audit_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

display(all_screen_summary[
    ["Interpretation", "Flagged count", "Flagged %", "Zero %", "Example rows"]
])
display(class_screen_summary)
print("Finished and saved. Original dataset unchanged.")

Checked 5,000,000 rows
Checked 10,000,000 rows
Checked 15,000,000 rows
Checked 20,000,000 rows
Checked 25,000,000 rows


,Interpretation,Flagged count,Flagged %,Zero %,Example rows
IN_BYTES,Potential IQR outlier,6088280,22.122901,0.000000,"[1074, 1383, 1384]"
OUT_BYTES,Potential IQR outlier,4611672,16.757371,43.016243,"[1383, 1547, 1573]"
IN_PKTS,Potential IQR outlier,4952626,17.996291,0.000000,"[1383, 1384, 1547]"
OUT_PKTS,Potential IQR outlier,4659346,16.930603,43.016243,"[1383, 1384, 1547]"
FLOW_DURATION_MILLISECONDS,Potential IQR outlier,5872001,21.337011,0.000000,"[661, 724, 735]"
DURATION_IN,Different from common value; IQR=0,5916637,21.499205,78.500795,"[52, 236, 349]"
DURATION_OUT,Different from common value; IQR=0,4739456,17.221698,82.778302,"[1383, 1384, 1547]"
LONGEST_FLOW_PKT,Potential IQR outlier,5903741,21.452345,0.000000,"[1383, 1384, 1547]"
SHORTEST_FLOW_PKT,Potential IQR outlier,5946512,21.607761,0.000000,"[1074, 1621, 1622]"
MIN_IP_PKT_LEN,Potential IQR outlier,1346,0.004891,43.016025,"[503015, 503038, 503040]"


,Total flows,IQR-flagged flows,IQR=0 variation flows,IQR-flagged % within class
Benign,16792214,6958615,3008121,41.439533
scanning,1358977,103883,52916,7.644206
dos,203456,202224,183649,99.394464
injection,381777,373159,362046,97.742661
ddos,4141256,1088697,992979,26.289053
password,1594777,1530308,1523193,95.957491
xss,2834435,2805434,2175661,98.976833
ransomware,3971,3871,3810,97.481743
Backdoor,203384,191618,190814,94.214884
mitm,6013,6007,5727,99.900216


Finished and saved. Original dataset unchanged.


In [ ]:
zero_iqr_review = all_screen_summary.loc[
    all_screen_summary["IQR"] == 0,
    ["Q1", "Q3", "Zero %", "Flagged %", "Example rows"]
].rename(columns={
    "Flagged %": "Different from common value (%)"
})

display(zero_iqr_review)

,Q1,Q3,Zero %,Different from common value (%),Example rows
DURATION_IN,0.0,0.0,78.500795,21.499205,"[52, 236, 349]"
DURATION_OUT,0.0,0.0,82.778302,17.221698,"[1383, 1384, 1547]"
RETRANSMITTED_IN_BYTES,0.0,0.0,98.439797,1.560203,"[14606, 23528, 36110]"
RETRANSMITTED_IN_PKTS,0.0,0.0,98.439797,1.560203,"[14606, 23528, 36110]"
RETRANSMITTED_OUT_BYTES,0.0,0.0,98.970998,1.029002,"[836621, 836652, 838807]"
RETRANSMITTED_OUT_PKTS,0.0,0.0,98.970998,1.029002,"[836621, 836652, 838807]"
NUM_PKTS_128_TO_256_BYTES,0.0,0.0,92.671065,7.328935,"[1547, 1573, 1600]"
NUM_PKTS_256_TO_512_BYTES,0.0,0.0,85.930794,14.069206,"[1383, 1384, 1547]"
NUM_PKTS_512_TO_1024_BYTES,0.0,0.0,92.038651,7.961349,"[1383, 1384, 2162]"
NUM_PKTS_1024_TO_1514_BYTES,0.0,0.0,91.022559,8.977441,"[1383, 1384, 1547]"
